# TRAINING BERT LARGE

## SETTING UP THE ENVIRONMENT

In [1]:
!pip install -q evaluate seqeval huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

In [ ]:
import os

os.environ["HF_TOKEN"] = ""

Importing the libraries:

In [ ]:
import torch
import spacy
import evaluate

import numpy as np

from spacy    import displacy
from datasets import Value       , \
                     Sequence    , \
                     Features    , \
                     ClassLabel  , \
                     DatasetDict , \
                     load_dataset, \
                     concatenate_datasets

from transformers import Trainer                    , \
                         AutoConfig                 , \
                         AutoTokenizer              , \
                         TrainingArguments          , \
                         TokenClassificationPipeline, \
                         AutoModelForTokenClassification

Load the dataset:

In [ ]:
sources = {
    "wikiann_pt" : ("wikiann"            , "pt"),
    "lener_br"   : ("lfcc/portuguese_ner", None),
}

datasets = {name : load_dataset(repo, subset) \
                   if   subset
                   else load_dataset(repo)
            for name, (repo, subset) in sources.items()}
datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pt/validation-00000-of-00001.parquet:   0%|          | 0.00/636k [00:00<?, ?B/s]

pt/test-00000-of-00001.parquet:   0%|          | 0.00/628k [00:00<?, ?B/s]

pt/train-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/266k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/930 [00:00<?, ? examples/s]

{'wikiann_pt': DatasetDict({
     validation: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 10000
     })
     test: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 10000
     })
     train: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 20000
     })
 }),
 'lener_br': DatasetDict({
     train: Dataset({
         features: ['tokens', 'ner_tags'],
         num_rows: 3716
     })
     test: Dataset({
         features: ['tokens', 'ner_tags'],
         num_rows: 930
     })
 })}

In [6]:
features   = datasets["wikiann_pt"]["train"   ].features
label_list = features["ner_tags"              ].feature.names

num_labels = len(label_list)

id2label = {i     : label for i, label in enumerate(label_list)}
label2id = {label : i     for i, label in enumerate(label_list)}

print("Labels:", ", ".join(label_list))

Labels: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC


In [7]:
def map_label(original):
    o = original.upper()

    if "PESSOA" in o or \
       "PER"    in o:
        if o.startswith("I-"):
            return "I-PER"

        return "B-PER"

    if "ORG"         in o or \
       "ORGANIZACAO" in o or \
       "ORGANIZAÇÃO" in o:
        if o.startswith("I-"):
            return "I-ORG"

        return "B-ORG"

    if "LOC"   in o or \
       "LOCAL" in o:
        if o.startswith("I-"):
            return "I-LOC"

        return "B-LOC"

    return "O"


def normalize_dataset(ds):
    train_features = ds["train"].features

    def convert(example):
        original_tags = example["ner_tags"]

        if isinstance(train_features["ner_tags"].feature, ClassLabel):
            names           = train_features["ner_tags"].feature.names
            original_labels = [names[t] for t in original_tags]
        else:
            original_labels = original_tags

        new_tags = [label2id[map_label(lbl)] for lbl in original_labels]

        return {
            "tokens"   : example["tokens"],
            "ner_tags" : new_tags
        }

    new_splits = {}
    for split in ["train", "validation", "test"]:
        if split in ds:
            new_splits[split] = ds[split].map(
                convert,
                remove_columns=ds[split].column_names
            )

    features = Features({
        "tokens"   : Sequence(Value("string")),
        "ner_tags" : Sequence(Value("int64" ))
    })

    for split in new_splits:
        new_splits[split] = new_splits[split].cast(features)

    return DatasetDict(new_splits)


normalized = []
for name, ds in datasets.items():
    print()
    print(f"Normalizing {name}...")

    normalized.append(normalize_dataset(ds))


def concat(split):
    parts = [
        ds[split]
        for ds in normalized
        if split in ds
    ]

    return concatenate_datasets(parts)


print()
print("Concatenating...")

train = concat("train"     )
val   = concat("validation")
test  = concat("test"      )

dataset = DatasetDict({
    "train"      : train,
    "validation" : val  ,
    "test"       : test ,
})


print()
print("Dataset:")
print(dataset)


Normalizing wikiann_pt...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3716 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]


Concatenating...

Dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 23716
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10930
    })
})


Tokenize and align the dataset:

In [ ]:
model_name = "neuralmind/bert-large-portuguese-cased"

tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation         =True,
        padding   ="max_length",
        max_length=128
    )

    word_ids = tokenized.word_ids()
    labels   = example["ner_tags"]

    aligned = []
    for w in word_ids:
        if w is None:
            aligned.append(-100)
        else:
            aligned.append(labels[w])

    tokenized["labels"] = aligned

    return tokenized

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=False,
    remove_columns=["tokens"  ,
                    "ner_tags"]
)

Map:   0%|          | 0/23716 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10930 [00:00<?, ? examples/s]

In [ ]:
example = {
    "tokens"   : ["João", "comprou", "uma", "maçã", "vermelha"],
    "ner_tags" : [1, 0, 0, 2, 2]
}

tok = tokenize_and_align(example)

print("Original tokens:")
print(example ["tokens"])
print("Original NER tags:")
print(example ["ner_tags"])

print()

print("Subtokens:")
print(tokenizer.convert_ids_to_tokens(tok["input_ids"])[:10])
print("Word ids:"      )
print(tok.word_ids()[:10])
print("Aligned labels:")
print(tok ["labels"][:10])

Original tokens:
['João', 'comprou', 'uma', 'maçã', 'vermelha']
Original NER tags:
[1, 0, 0, 2, 2]

Subtokens:
['[CLS]', 'João', 'comprou', 'uma', 'maç', '##ã', 'vermelha', '[SEP]', '[PAD]', '[PAD]']
Word ids:
[None, 0, 1, 2, 3, 3, 4, None, None, None]
Aligned labels:
[-100, 1, 0, 0, 2, 2, 2, -100, -100, -100]


Auxiliary functions:

In [11]:
seqeval = evaluate.load("seqeval")

def decode_predictions(predictions, labels):
    preds = np.argmax(predictions, axis=-1)
    pred_strings  = []
    label_strings = []

    for pred_ids, label_ids in zip(preds, labels):
        p_str = []
        l_str = []
        for p, l in zip(pred_ids, label_ids):
            if l == -100:
                continue

            p_str.append(label_list[p])
            l_str.append(label_list[l])

        pred_strings .append(p_str)
        label_strings.append(l_str)

    return pred_strings, label_strings


def compute_metrics(eval_pred):
    preds, refs = decode_predictions(*eval_pred)
    results     = seqeval.compute(
        predictions=preds,
        references =refs
    )

    return {
        "precision" : results["overall_precision"],
        "recall"    : results["overall_recall"   ],
        "f1"        : results["overall_f1"       ],
        "accuracy"  : results["overall_accuracy" ],
    }


def predict_ner(model, tokens):
    device   = next(model.parameters()).device
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors     ="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)

    preds    = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = encoding.word_ids()

    results = []
    prev    = None
    for p, w in zip(preds, word_ids):
        if w is None or w == prev:
            continue

        results.append((
            tokens  [w],
            id2label[p]
        ))

        prev = w

    return results

## BERT LARGE

In [12]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label  =id2label  ,
    label2id  =label2id  ,
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name, config=config, ignore_mismatched_sizes=True
)

training_args = TrainingArguments(
    output_dir   ="./bertimbau-large-wikiann-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=   5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size =32,
    num_train_epochs=5   ,
    weight_decay    =0.01,
    report_to="none",
    optim    ="adamw_torch"
)

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
for param in model.base_model.parameters():
    param.requires_grad = False

num_layers  = len(model.base_model.encoder.layer)
start_layer = num_layers // 3

print(f"Total layers: {num_layers}")
print(f"Unfreezing layers from: {start_layer} to {num_layers - 1}")

for layer_idx in range(start_layer, num_layers):
    for param in model.base_model.encoder.layer[layer_idx].parameters():
        param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

Total layers: 24
Unfreezing layers from: 8 to 23


In [ ]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters       : {total_params    :,}")
print(f"Trainable parameters   : {trainable_params:,}")
print(f"Non-trainable params   : {total_params - trainable_params:,}")

Total parameters       : 333,353,991
Trainable parameters   : 201,546,759
Non-trainable params   : 131,807,232


In [15]:
trainer = Trainer(
    model=model        ,
    args =training_args,
    train_dataset=tokenized_dataset["train"     ],
    eval_dataset =tokenized_dataset["validation"],
    compute_metrics =compute_metrics,
    processing_class=tokenizer      ,
)

print("Device:", trainer.args.device)

Device: cuda:0


In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.254600,0.192418,0.893243,0.895508,0.894374,0.945917
2,0.132300,0.180785,0.888399,0.909134,0.898647,0.948940
3,0.061100,0.180999,0.910961,0.917344,0.914141,0.955714
4,0.045000,0.198984,0.914440,0.922459,0.918432,0.956949
5,0.019800,0.215735,0.915825,0.923619,0.919706,0.957724


TrainOutput(global_step=3710, training_loss=0.0872335219318976, metrics={'train_runtime': 7848.1767, 'train_samples_per_second': 15.109, 'train_steps_per_second': 0.473, 'total_flos': 2.753198551251456e+16, 'train_loss': 0.0872335219318976, 'epoch': 5.0})

In [17]:
from pprint import pprint

test_metrics = trainer.evaluate(tokenized_dataset["test"])

print ("Test:")
pprint(test_metrics)

Test:
{'epoch': 5.0,
 'eval_accuracy': 0.9666724227249179,
 'eval_f1': 0.9329458539107489,
 'eval_loss': 0.18861310184001923,
 'eval_precision': 0.9292513141071175,
 'eval_recall': 0.9366698886621558,
 'eval_runtime': 248.1059,
 'eval_samples_per_second': 44.054,
 'eval_steps_per_second': 1.378}


In [18]:
trainer  .save_model     ("./bertimbau-large-wikiann-ner-model")
tokenizer.save_pretrained("./bertimbau-large-wikiann-ner-model")

('./bertimbau-large-wikiann-ner-model/tokenizer_config.json',
 './bertimbau-large-wikiann-ner-model/special_tokens_map.json',
 './bertimbau-large-wikiann-ner-model/vocab.txt',
 './bertimbau-large-wikiann-ner-model/added_tokens.json',
 './bertimbau-large-wikiann-ner-model/tokenizer.json')

In [19]:
nlp_ner = TokenClassificationPipeline(
    model    =model    ,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if "cuda" in str(next(model.parameters()).device) else -1
)

def ner_displacy(text):
    ner_results = nlp_ner(text)

    docs = [{
        "text": text,
        "ents": [
            {
                "start" : ent["start"       ],
                "end"   : ent["end"         ],
                "label" : ent["entity_group"]
            }
            for ent in ner_results
        ],
    }]

    colors = {
        "PER": "linear-gradient(90deg, #999999, #cccccc)",
        "LOC": "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
        "ORG": "linear-gradient(90deg, #ffcc70, #ff9a3c)",
    }

    options = {"ents": ["PER", "LOC", "ORG"], "colors": colors}

    return docs, options

Device set to use cuda:0


In [20]:
test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."
]

for i, text in enumerate(test_texts, 1):
    docs, options = ner_displacy(text)
    displacy.render(docs, style="ent", options=options, manual=True, jupyter=True)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
